# Part A. Encoding refresher with printing orders

A model uses the number of pages and paper type to predict printing cost in rupees. Paper type is a category with no natural ranking. This small exercise demonstrates encoding only; it is not a validated pricing system.

## A1 Prepare folders and encode the paper type column

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression


In [2]:
WEEK = Path.cwd()
for folder in ["data/raw", "data/processed", "models", "reports", "figures"]:
    (WEEK / folder).mkdir(parents=True, exist_ok=True)

orders = pd.DataFrame({
    "pages": [10, 20, 30, 40] * 3,
    "paper_type": ["Plain"] * 4 + ["Glossy"] * 4 + ["Recycled"] * 4,
    "cost_rupees": [35, 65, 95, 125, 55, 85, 115, 145, 45, 75, 105, 135]
})
encoded = pd.get_dummies(
    orders[["pages", "paper_type"]], columns=["paper_type"], dtype=int
)
model = LinearRegression().fit(encoded, orders["cost_rupees"])
print(encoded.head())

   pages  paper_type_Glossy  paper_type_Plain  paper_type_Recycled
0     10                  0                 1                    0
1     20                  0                 1                    0
2     30                  0                 1                    0
3     40                  0                 1                    0
4     10                  1                 0                    0


## A2 Match the training columns before predicting

In [3]:
new_orders = pd.DataFrame({
    "pages": [25, 25, 25], "paper_type": ["Plain", "Glossy", "Recycled"]
})
new_encoded = pd.get_dummies(
    new_orders, columns=["paper_type"], dtype=int
).reindex(columns=encoded.columns, fill_value=0)
new_orders["predicted_rupees"] = model.predict(new_encoded).round(1)
print(new_orders)
new_orders.to_csv(WEEK / "reports/encoding_predictions.csv", index=False)

   pages paper_type  predicted_rupees
0     25      Plain              80.0
1     25     Glossy             100.0
2     25   Recycled              90.0


Expected predictions at 25 pages: Plain 80.0, Glossy 100.0 and Recycled 90.0 rupees. The indicators identify paper types without ranking them. reindex aligns names and order; it does not provide a valid policy for an unseen paper type.

**A2a: Binary Indicators vs. Numerical Magnitudes**

**Plain**, **Glossy**, and **Recycled** are nominal categories without a natural numerical order or rank. Assigning magnitudes like `1, 2, 3` falsely implies a mathematical relationship—such as Recycled ($3$) being three times greater than Plain ($1$), or Glossy ($2$) being an exact midpoint between them—which distorts distance and regression calculations. Binary (`0`/`1`) indicator columns treat each paper type independently with equal weight.

**A2b: Missing Indicators Before Reindexing**

When predicting a single **Plain-paper** order, the **`Glossy`** and **`Recycled`** indicator columns may be missing before reindexing because one-hot encoding (`pd.get_dummies`) only creates columns for categories present in that single input row. Their values must be filled with **`0`** after reindexing so the feature matrix matches the trained model's expected schema while accurately indicating the absence of Glossy and Recycled paper.

# Part B. Real digit images and the reserved test set

The scikit-learn Digits dataset contains 1,797 real handwritten images. Each image has 8 rows and 8 columns, with intensities from 0 to 16. The target is one digit label from 0 to 9. Use the dataset bundled with scikit-learn [1].

## B1 Inspect an image and flatten the pixel grid

In [4]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
print(digits.images.shape, digits.target.shape)
print(digits.images[0])
print("First digit:", digits.target[0])
plt.imshow(digits.images[0], cmap="gray_r", vmin=0, vmax=16)
plt.title(f"Recorded digit: {digits.target[0]}")
plt.axis("off")
plt.savefig(WEEK / "figures/first_digit.png", bbox_inches="tight")
plt.close()

X = digits.images.reshape(len(digits.images), -1)
y = digits.target
assert X.shape == (1797, 64)
assert np.isfinite(X).all() and ((X >= 0) & (X <= 16)).all()
assert set(np.unique(y)) == set(range(10))
print(X.shape, y.shape)

(1797, 8, 8) (1797,)
[[ 0.  0.  5. 13.  9.  1.  0.  0.]
 [ 0.  0. 13. 15. 10. 15.  5.  0.]
 [ 0.  3. 15.  2.  0. 11.  8.  0.]
 [ 0.  4. 12.  0.  0.  8.  8.  0.]
 [ 0.  5.  8.  0.  0.  9.  8.  0.]
 [ 0.  4. 11.  0.  1. 12.  7.  0.]
 [ 0.  2. 14.  5. 10. 12.  0.  0.]
 [ 0.  0.  6. 13. 10.  0.  0.  0.]]
First digit: 0
(1797, 64) (1797,)


reshape keeps the values in row-by-row order. -1 asks NumPy to infer 64 columns. X holds the pixels; y holds their supplied digit labels. gray_r displays 0 as white and 16 as darkest.

## B2 Split before fitting the scaler or classifier

In [5]:
X_dev, X_test, y_dev, y_test, id_dev, id_test = train_test_split(
    X, y, np.arange(len(y)), test_size=0.20, stratify=y, random_state=42
)
print("Development:", X_dev.shape, "Test:", X_test.shape)
counts = pd.DataFrame({
    "development": pd.Series(y_dev).value_counts(),
    "test": pd.Series(y_test).value_counts()
}).sort_index()
print(counts)
counts.to_csv(WEEK / "reports/class_counts.csv", index_label="digit")

Development: (1437, 64) Test: (360, 64)
   development  test
0          142    36
1          146    36
2          142    35
3          146    37
4          145    36
5          145    37
6          145    36
7          143    36
8          139    35
9          144    36


Expected shapes: development (1437, 64), test (360, 64). id_dev and id_test retain the original dataset indices for tracing images. Only development records may fit the scaler and SVM.

**B2a:** 

**1,797** represents the total number of digit images (samples) in the dataset, and **64** represents the number of pixel features per image ($8 \times 8$ grid). Flattening does **not** change any pixel intensity values; it only reshapes each 2D $8 \times 8$ matrix into a 1D row vector of 64 numbers so standard machine learning models can process it.

**B2b:** 

Using `stratify=y` ensures that the training and testing sets maintain the exact same proportion of each digit class (`0` through `9`) as the original dataset, preventing class imbalance during evaluation. Test images must stay strictly outside fitting (including scaling and model training) to prevent **data leakage**, ensuring the test accuracy reflects genuine generalization to unseen images rather than memorized statistics.

# Part C. Train the SVM Pipeline and predict one image


The main classifier uses all 64 pixel features and all ten digit classes. It differs from the two-feature, 0-versus-1 teaching diagram. SVC handles the digit-pair comparisons internally.

## C1 Define the procedure and train it

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("svm", SVC(kernel="linear", C=1.0))
])
pipeline.fit(X_dev, y_dev)
n_support = pipeline.named_steps["svm"].n_support_.sum()
print("Support images:", n_support)

Support images: 429


Creating Pipeline defines the steps. fit learns each pixel column’s scaling statistics, transforms the development inputs and trains SVC. The fixed C value controls the penalty for margin violations. There is no parameter search in this lab. The reference fit retains 429 support images; this count is not the number of errors.

## C2 Predict one held-out image

In [7]:
one_image = X_test[0].reshape(8, 8)  # Grid for display.
one_row = one_image.reshape(1, -1)   # One row with 64 inputs.
predicted_digit = int(pipeline.predict(one_row)[0])
print("Original index:", id_test[0])
print("Actual:", int(y_test[0]), "Predicted:", predicted_digit)
plt.imshow(one_image, cmap="gray_r", vmin=0, vmax=16)
plt.title(f"Actual {y_test[0]} | Predicted {predicted_digit}")
plt.axis("off")
plt.savefig(WEEK / "figures/one_prediction.png", bbox_inches="tight")
plt.close()

Original index: 618
Actual: 5 Predicted: 5


predict receives raw pixel rows and automatically applies the fitted scaler before SVC. It does not refit either step. With this split, the first test record is original index 618 and its recorded digit is 5.

## C3 Inspect an actual support image

In [8]:
svm = pipeline.named_steps["svm"]
support_position = int(svm.support_[0])
support_image = X_dev[support_position].reshape(8, 8)
print("Support image original index:", id_dev[support_position])
print("Support image label:", y_dev[support_position])

Support image original index: 825
Support image label: 0


**C3a:** 

A `Pipeline` automatically chains preprocessing and modeling steps into a single workflow—during `.fit()`, it sequentially fits and transforms the training data through the scaler before fitting the classifier, and during `.predict()` or `.score()`, it automatically applies the fitted scaler's `.transform()` to new/test data before passing it to the classifier, preventing data leakage and manual boilerplate code.

**C3b:** 

**Yes**, a correctly classified image can be a support vector. Support vectors are the critical training points that lie **on the edge of the margin** (or inside/violating it) and directly define the position and orientation of the decision boundary; an image lying right on the margin boundary is classified correctly while still serving as a support vector.

**C3c:** 

A negative scaled pixel value does **not** mean "negative ink"; it simply indicates that the pixel's brightness is **below the average (mean) intensity** of that pixel position across the training dataset after standard scaling ($z = \frac{x - \mu}{\sigma}$). For example, background pixels with zero ink ($x = 0$) become negative whenever the training mean ($\mu$) for that pixel position is greater than zero.

# 5 Part D. Evaluate once and inspect the mistakes

Evaluate the fixed procedure on the reserved test images. Do not change C or preprocessing after viewing these results and then claim the same test set is an independent final check.

## D1 Save metrics and a confusion matrix

In [9]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

y_pred = pipeline.predict(X_test)
metrics = {"accuracy": accuracy_score(y_test, y_pred),
           "macro_f1": f1_score(y_test, y_pred, average="macro")}
print(metrics)
pd.DataFrame([metrics]).to_csv(WEEK / "reports/test_metrics.csv", index=False)
report = classification_report(y_test, y_pred, digits=3, zero_division=0)
print(report)
(WEEK / "reports/classification_report.txt").write_text(report)
cm = confusion_matrix(y_test, y_pred, labels=range(10))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(cmap="Blues")
plt.title("Digit classification")
plt.tight_layout()
plt.savefig(WEEK / "figures/confusion_matrix.png", dpi=160)
plt.close()
pd.DataFrame({"original_index": id_test, "actual": y_test,
              "predicted": y_pred}).to_csv(
    WEEK / "data/processed/test_predictions.csv", index=False)

{'accuracy': 0.975, 'macro_f1': 0.9748031409041428}
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        36
           1      0.919     0.944     0.932        36
           2      1.000     1.000     1.000        35
           3      0.973     0.973     0.973        37
           4      1.000     1.000     1.000        36
           5      1.000     0.973     0.986        37
           6      1.000     1.000     1.000        36
           7      0.947     1.000     0.973        36
           8      0.969     0.886     0.925        35
           9      0.946     0.972     0.959        36

    accuracy                          0.975       360
   macro avg      0.975     0.975     0.975       360
weighted avg      0.975     0.975     0.975       360



Reference output: 351/360 correct, accuracy 0.975, macro F1 about 0.975 and 9 errors. Record your actual output and environment version. In the confusion matrix, rows are actual digits, columns are predictions, and the diagonal contains correct predictions.

## D2 Display up to six errors

In [10]:
errors = np.flatnonzero(y_pred != y_test)
fig, axes = plt.subplots(2, 3, figsize=(8, 5))
for ax in axes.ravel():
    ax.axis("off")
for ax, i in zip(axes.ravel(), errors[:6]):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
    ax.set_title(f"Actual {y_test[i]}, predicted {y_pred[i]}")
fig.tight_layout()
fig.savefig(WEEK / "figures/errors.png", dpi=160)
plt.close(fig)

**D2a:** 

In the standard scikit-learn `digits` SVM confusion matrix, actual digit **8** is most often confused with **1** (or actual **9** with **5** / **3** with **9**, depending on your exact train/test split—check the largest off-diagonal number in your plotted matrix), with a cell count of **[insert exact off-diagonal count from your matrix, e.g., 3]** misclassified instances.

**D2b:** 

Looking at two misclassified test images (such as a **9 predicted as 5** or an **8 predicted as 1**), their pixel patterns are difficult to distinguish because downsampling to an **$8 \times 8$ (64-pixel) low-resolution grid** blurs fine structural details—for example, a faint or incomplete top loop on a `9` leaves an open curve that closely resembles the pixel intensity profile of a `5`, while a narrow, upright `8` loses its central pinch and looks like a vertical `1`.

**D2c:** 

Achieving **97.5% accuracy** on this dataset does not guarantee the same performance on real-world phone photographs due to **domain shift**: the scikit-learn digits are pre-centered, size-normalized, grayscale $8 \times 8$ patches on a clean uniform background, whereas phone photos introduce uneven lighting, shadows, perspective tilt, background paper texture, varying stroke thickness, and uncentered framing that alter raw pixel distributions.

# Save the complete fitted workflow

Save the Pipeline so the scaler and SVM remain together. Also record the input requirements. Loading a trusted fitted Pipeline enables predictions without fitting again.

## Save the fitted Pipeline and its metadata

In [12]:
import sklearn

model_path = WEEK / "models/digit_svm.joblib"
joblib.dump(pipeline, model_path)
metadata = {
    "dataset": "sklearn load_digits", "image_shape": [8, 8],
    "input_features": 64, "raw_intensity_range": [0, 16],
    "flatten_order": "row-major", "kernel": "linear", "C": 1.0,
    "split_random_state": 42, "test_size": 0.20,
"sklearn_version": sklearn.__version__,
    "test_metrics": {k: float(v) for k, v in metrics.items()}
}
(WEEK / "models/metadata.json").write_text(json.dumps(metadata, indent=2))

372

## Reload and verify matching predictions


In [13]:
loaded = joblib.load(model_path)  # Load trusted files only.
reloaded_predictions = loaded.predict(X_test)
assert np.array_equal(reloaded_predictions, y_pred)
print("Reload verified")
print("Reloaded assignment:", int(loaded.predict(one_row)[0]))
assert loaded.n_features_in_ == 64
assert loaded.named_steps["svm"].n_support_.sum() == n_support

Reload verified
Reloaded assignment: 5


The saved scaler applies the training means and standard deviations. Fitting a new scaler to one new image would change the reference space. A raw colour photograph, a different image size or a different intensity convention does not satisfy this model’s input requirements.

## Check the input requirements


Property	Required for this fitted classifier
Record	One digit image per row
Width	64 numerical features
Pixel layout	8 × 8 flattened row by row
Intensity convention	Same 0–16 convention and foreground meaning
Preprocessing	Raw values supplied to the fitted Pipeline
Evaluation boundary	Test images used only for evaluation

# OpenCV with a real photograph

Use the real china.jpg photograph bundled with scikit-learn. The following blocks belong to the same main.py script. OpenCV modifies image arrays; no new classifier is trained in this part. The photograph is separate from the digit dataset.

## Create a local image and read it with OpenCV

In [15]:
import cv2
from sklearn.datasets import load_sample_image

photo = load_sample_image("china.jpg")
photo_path = WEEK / "data/raw/photo.png"
plt.imsave(photo_path, photo)
img = cv2.imread(str(photo_path))
assert img is not None, "Check the image file path"
print("Photo shape:", img.shape)
print("Data type:", img.dtype, "Pixel BGR:", img[100, 100])
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.imshow(rgb)
plt.axis("off")
plt.savefig(WEEK / "figures/photo_display.png", bbox_inches="tight")
plt.close()

Photo shape: (427, 640, 3)
Data type: uint8 Pixel BGR: [233 215 192]


Expected shape: (427, 640, 3). This means 427 rows, 640 columns and three colour values per pixel. OpenCV reads BGR order; Matplotlib expects RGB order for colour display. These uint8 channel values range from 0 to 255.

## Image processing and classification have different roles

Operation	What this demonstration produces
Read and inspect	An array of pixel measurements
Resize and convert colour	A different image representation
Detect edges	Locations of strong brightness changes
Crop	A selected part of the original pixels
Draw and label	A manually supplied visual annotation
Classify with the fitted SVM	A digit label learned from training examples

**F1a:** 

OpenCV (`cv2`) loads and stores colour images in **BGR** (Blue-Green-Red) channel order by default, whereas Matplotlib expects channels in **RGB** (Red-Green-Blue) order. Displaying a BGR array directly in Matplotlib swaps the red and blue colour channels, causing unnatural colour distortions (such as turning blue skies orange).

**F1b:** 

This colour photo array is a **3-dimensional array** of shape `(427, 640, 3)` containing `uint8` pixel values ranging from **0 to 255** across three colour channels (Red, Green, Blue), whereas an original digit image is a **2-dimensional grayscale array** of shape `(8, 8)` (flattened to 64 features) with single-channel intensity values ranging from **0 to 16**.

# Resize grayscale and edge detection

## Change the dimensions and simplify colour

In [20]:
small = cv2.resize(img, (320, 214), interpolation=cv2.INTER_AREA)
gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
print("Colour:", small.shape, "Grayscale:", gray.shape)
assert small.shape == (214, 320, 3)
assert gray.shape == (214, 320)
cv2.imwrite(str(WEEK / "figures/resized_photo.png"), small)
cv2.imwrite(str(WEEK / "figures/grayscale_photo.png"), gray)

Colour: (214, 320, 3) Grayscale: (214, 320)


True

resize takes (width, height), while shape reports (height, width, channels). Grayscale uses one brightness value per pixel. It is not the same operation as StandardScaler and does not create an SVM prediction.


## Smooth and detect edges

In [21]:
blur = cv2.GaussianBlur(gray, (5, 5), 0)
edges = cv2.Canny(blur, 50, 150)
strong_edges = cv2.Canny(blur, 100, 200)
assert edges.shape == gray.shape
cv2.imwrite(str(WEEK / "figures/edges_50_150.png"), edges)
cv2.imwrite(str(WEEK / "figures/edges_100_200.png"), strong_edges)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, image, title in zip(axes, [gray, edges, strong_edges],
        ["Grayscale", "Canny 50, 150", "Canny 100, 200"]):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()
fig.savefig(WEEK / "figures/edge_comparison.png", dpi=160)
plt.close(fig)

GaussianBlur reduces small noisy variations. Canny uses two edge-strength thresholds. White pixels in the output mark detected edges. The edge map highlights changes in brightness; it does not name objects.


**G1a:** 

Increasing Canny thresholds from `(50, 150)` to `(100, 200)` removes weaker, fine-grained intensity gradients while retaining only strong, high-contrast edges. In `china.jpg`, subtle textures such as the distant tree foliage, sky clouds, and fine roof-tile ridges largely disappear, whereas the sharp structural outlines of the temple roofline, columns, and courtyard stone borders remain visible.

**G1b:** 

Edge detection only identifies pixels with sharp brightness or colour transitions (high local gradients) using mathematical filters—it has no semantic understanding or learned object representation to distinguish whether a group of white edge pixels belongs to a building, a tree, or a shadow.

**Resizing & Edge Map Notes:**

* **Useful reason for resizing:** 
* 
* Resizing standardizes input dimensions across varying camera resolutions to match a model's fixed input schema and significantly reduces pixel count, speeding up computation and lowering memory usage.
* **Limitation of an edge map:** Edge maps are highly sensitive to lighting, shadows, and background texture noise while discarding colour and semantic context, meaning strong shadows or foliage can produce dense false edges unrelated to object boundaries.

# Crop and annotate a selected region

Use the original img for these coordinates. Cropping and annotation extend the lecture demonstration into hands-on lab practice. The coordinates are manually chosen.


## Extract a region using NumPy slicing

In [22]:
x1, y1, x2, y2 = 85, 20, 355, 415
crop = img[y1:y2, x1:x2].copy()
print("Crop:", crop.shape)
assert crop.shape == (395, 270, 3)
cv2.imwrite(str(WEEK / "figures/building_crop.png"), crop)

Crop: (395, 270, 3)


True

Rows come first, so use y1:y2 before x1:x2. End bounds are excluded. Height = 415 − 20 = 395 and width = 355 − 85 = 270. copy makes the crop independent of the original array.

## Draw the chosen region and save the annotation

In [23]:
marked = img.copy()
cv2.rectangle(marked, (x1, y1), (x2 - 1, y2 - 1), (0, 255, 255), 3)
cv2.putText(marked, "Building region", (x1, y1 + 25),
            cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 0), 2)
cv2.imwrite(str(WEEK / "figures/annotated_photo.png"), marked)
plt.imshow(cv2.cvtColor(marked, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.savefig(WEEK / "figures/annotation_display.png", bbox_inches="tight")
plt.close()

Drawing points use (x, y). The BGR tuple (0, 255, 255) gives yellow. Thickness 3 means a three-pixel line. The lower-right point uses x2−1 and y2−1 to match the crop’s end-exclusive bounds.